# Monte Carlo and Scaling Studies

We repeat sprinklings to estimate mean and spread of observables, and then
see how those observables scale with $N$. To keep this notebook robust on
Windows, we avoid multiprocessing and run a small number of trials.

In [ ]:
from pathlib import Path
import sys
import importlib.util

cwd = Path.cwd()
if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError("Could not locate project root (expected a src/ folder).")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

HAS_SCIPY = importlib.util.find_spec("scipy") is not None
print("scipy available:", HAS_SCIPY)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.core_operations import sprinkle, causal_matrix

try:
    from src import observables as obs
    HAS_OBS = True
except Exception as exc:
    HAS_OBS = False
    print("Observables import failed (likely missing scipy):", exc)

dim = 2
N_list = [30, 50, 80]
trials = 5

results = {
    'N': [],
    'r_mean': [], 'r_std': [],
    'd_mean': [], 'd_std': [],
    'L_mean': [], 'L_std': [],
    'AC_mean': [], 'AC_std': []
}

if HAS_OBS:
    for N in N_list:
        r_list, d_list, L_list, AC_list = [], [], [], []
        for _ in range(trials):
            pts = sprinkle(N, dim=dim)
            R = causal_matrix(pts, dim=dim)
            r = obs.ordering_fraction(R)
            d = obs.estimate_dimension(r)
            L = obs.longest_chain_length(R)
            AC = obs.largest_antichain(R)
            r_list.append(r)
            if d is not None:
                d_list.append(d)
            L_list.append(L)
            AC_list.append(AC)

        results['N'].append(N)
        results['r_mean'].append(float(np.mean(r_list)))
        results['r_std'].append(float(np.std(r_list)))
        results['d_mean'].append(float(np.mean(d_list)) if d_list else float('nan'))
        results['d_std'].append(float(np.std(d_list)) if d_list else float('nan'))
        results['L_mean'].append(float(np.mean(L_list)))
        results['L_std'].append(float(np.std(L_list)))
        results['AC_mean'].append(float(np.mean(AC_list)))
        results['AC_std'].append(float(np.std(AC_list)))

    print(results)
else:
    print("Skipping Monte Carlo because observables are unavailable.")

## Scaling Plots
The plots below visualize how each observable changes with $N$.
With small $N$, the trends are noisy but still illustrate the idea.

In [ ]:
if HAS_OBS and results['N']:
    fig, axs = plt.subplots(2, 2, figsize=(10, 8))
    axs = axs.flatten()

    axs[0].errorbar(results['N'], results['r_mean'], yerr=results['r_std'], fmt='o-')
    axs[0].set_title('Ordering Fraction vs N')
    axs[0].set_xlabel('N')
    axs[0].set_ylabel('r')
    axs[0].grid(True, linestyle=':')

    axs[1].errorbar(results['N'], results['d_mean'], yerr=results['d_std'], fmt='o-', color='green')
    axs[1].set_title('Estimated Dimension vs N')
    axs[1].set_xlabel('N')
    axs[1].set_ylabel('d')
    axs[1].grid(True, linestyle=':')

    axs[2].errorbar(results['N'], results['L_mean'], yerr=results['L_std'], fmt='o-', color='orange')
    axs[2].set_title('Longest Chain vs N')
    axs[2].set_xlabel('N')
    axs[2].set_ylabel('L')
    axs[2].grid(True, linestyle=':')

    axs[3].errorbar(results['N'], results['AC_mean'], yerr=results['AC_std'], fmt='o-', color='red')
    axs[3].set_title('Largest Antichain vs N')
    axs[3].set_xlabel('N')
    axs[3].set_ylabel('AC')
    axs[3].grid(True, linestyle=':')

    plt.tight_layout()
    plt.show()
else:
    print("Plots skipped (observables unavailable).")